# 2ème partie du projet

In [4]:
# Création de la base Faiss et de l'index

from pathlib import Path

import faiss
import numpy as np
import pandas as pd

# Colonnes à retourner avec les résultats de recherche.
colonnes_metadata_candidates = [
    "uid",
    "title",
    "description",
    "longDescription",
    "keywords",
    "firstTiming",
    "nextTiming",
    "lastTiming",
    "location.name",
    "location.address",
    "location.city",
    "location.region",
    "location.adminLevel1",
    "location.latitude",
    "location.longitude",
    "canonicalUrl",
    "agenda_uid_source",
    "agenda_titre_source",
]

df_embeddings = pd.read_parquet("data/parquet_sortie/evenements_embeddings_mistral.parquet")
df_nettoye = pd.read_parquet("data/parquet_sortie/evenements_culturels_nettoye.parquet")

colonnes_metadata = [
    colonne
    for colonne in colonnes_metadata_candidates
    if colonne in df_nettoye.columns
]

# Traitement des métadonnées
# Associe chaque chunk à l'événement dont il provient.
metadata_evenements = df_nettoye[colonnes_metadata].copy()
metadata_evenements["index_evenement"] = metadata_evenements.index

metadata_faiss = (
    df_embeddings[["index_evenement", "numero_chunk", "texte_chunk"]]
    .merge(
        metadata_evenements,
        on="index_evenement",
        how="left",
        validate="many_to_one",
    )
    .reset_index(drop=True)
)